# Experiment 1 Rerun — Only Well-Documented Poets

This is the flip side of the earlier "size-matched" test (Q6). That test
restricted to MINOR poets (output comparable to one surah) and found the
overlap with the Quran got stronger, not weaker — suggesting sparse-data
poets might be driving the effect.

This notebook does the opposite: keeps ONLY poets whose total surviving
output is LARGER than the biggest surah (286 verses) — i.e., only the
well-documented poets with enough text to have a well-defined style
signature — then reruns Experiment 1 (Quran as one entity vs. this
restricted poet pool).

**If the sample-size explanation is right**, the Quran should now look
more clearly separate from all poets (fewer/no poets sharing its
cluster, or it becomes a clean outlier). If the overlap persists here
too, that argues against the sample-size explanation.

**Threshold used:** total verses > 286 (the largest surah's verse
count) — the direct complement of the Q6 filter, so the two results are
a clean before/after comparison.

**Embeddings are cached** (`embed_cache/`) — if you run this in the same
folder as the earlier extended-analysis notebook, it reuses those cached
vectors and skips straight to the new clustering (a few minutes instead
of 30+).

**Before you start:** put `poems.db` in the same folder as this
notebook.

Run cells top to bottom, **Shift+Enter**.

In [1]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn scipy requests


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2 -- Configuration (same values as before, plus embedding cache paths)
import re, sqlite3, hashlib, json, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

DB_PATH = Path("poems.db")
CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU (slower).")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print("Config loaded.")

GPU available: NVIDIA GeForce RTX 4080 SUPER
Config loaded.


In [3]:
# CELL 3 -- Load the 260 poets from poems.db
conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query(
    "SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count "
    "FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL "
    "AND LENGTH(poem_text) >= 50",
    conn,
)
conn.close()

df["poem_hash"] = df["poem_text"].apply(lambda t: hashlib.md5(t.strip().encode("utf-8")).hexdigest())
df = df.drop_duplicates(subset=["poem_hash"]).copy()

poems_by_poet = defaultdict(list)
for _, row in df.iterrows():
    poems_by_poet[row["poet_name"]].append(row["poem_text"])
poems_by_poet = dict(poems_by_poet)

poet_total_verses = {
    poet: sum(len(split_verses(p)) for p in poems)
    for poet, poems in poems_by_poet.items()
}

print(f"Poets loaded: {len(poems_by_poet)}")
print(f"Poems loaded: {len(df)}")

Poets loaded: 260
Poems loaded: 2328


In [4]:
# CELL 4 -- Fetch Quran text + per-surah size metadata
cache_file = CACHE_DIR / "quran_ayat.json"

if cache_file.exists():
    print("Loading Quran from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
surah_poem_text = {}
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    ayat_texts = [a["text"] for a in s["ayahs"]]
    surah_poem_text[snum] = "\n".join(ayat_texts)

surah_verse_count = {snum: len(split_verses(text)) for snum, text in surah_poem_text.items()}

print(f"Surahs loaded: {len(surah_poem_text)}")
print(f"Surah verse-count range: {min(surah_verse_count.values())} to {max(surah_verse_count.values())}")

Fetching Quran from Al Quran Cloud API...
Fetched and cached.
Surahs loaded: 114
Surah verse-count range: 3 to 286


In [5]:
# CELL 5 -- Embed the 260 poets (cached after first run)
from sentence_transformers import SentenceTransformer

poet_cache_file = EMBED_CACHE_DIR / "poet_embeddings.pkl"

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded. Embedding dim:", model.get_sentence_embedding_dimension())

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average(poems_by_author, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    out = {}
    rng = np.random.RandomState(seed)
    for author, poems in poems_by_author.items():
        poem_vectors = []
        for poem in poems:
            verses = split_verses(poem)
            if len(verses) > max_verses:
                idx = rng.choice(len(verses), max_verses, replace=False)
                verses = [verses[i] for i in sorted(idx)]
            if verses:
                poem_vectors.append(np.mean(_encode(verses), axis=0))
        if poem_vectors:
            out[author] = np.mean(poem_vectors, axis=0)
    return out

if poet_cache_file.exists():
    print("Loading cached poet embeddings...")
    with open(poet_cache_file, "rb") as f:
        poet_embeddings = pickle.load(f)
    print(f"Loaded {len(poet_embeddings)} cached poet embeddings.")
else:
    print("Embedding 260 poets (this is the slow step, one-time only)...")
    poet_embeddings = embed_poem_verse_average(poems_by_poet)
    with open(poet_cache_file, "wb") as f:
        pickle.dump(poet_embeddings, f)
    print(f"Done and cached. {len(poet_embeddings)} poets embedded.")

Loading model: akhooli/Arabic-SBERT-100K


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded. Embedding dim: 768
Embedding 260 poets (this is the slow step, one-time only)...
Done and cached. 260 poets embedded.


In [6]:
# CELL 6 -- Embed the 114 surahs (cached after first run)
surah_cache_file = EMBED_CACHE_DIR / "surah_vectors.pkl"

if surah_cache_file.exists():
    print("Loading cached surah embeddings...")
    with open(surah_cache_file, "rb") as f:
        surah_vectors = pickle.load(f)
    print(f"Loaded {len(surah_vectors)} cached surah embeddings.")
else:
    print("Embedding 114 surahs...")
    surah_vectors = {}
    for snum, poem_text in surah_poem_text.items():
        verses = split_verses(poem_text)
        if verses:
            surah_vectors[snum] = np.mean(_encode(verses), axis=0)
    with open(surah_cache_file, "wb") as f:
        pickle.dump(surah_vectors, f)
    print(f"Done and cached. {len(surah_vectors)} surahs embedded.")

quran_whole_vector = np.mean(list(surah_vectors.values()), axis=0)
print("Combined whole-Quran vector computed.")

Embedding 114 surahs...
Done and cached. 114 surahs embedded.
Combined whole-Quran vector computed.


## Filter to well-documented poets, then rerun Experiment 1

In [7]:
# CELL 7 -- Filter to well-documented poets, rerun Experiment 1
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])

    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, n - 1), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)

    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None

    result_df = pd.DataFrame({
        "name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1],
    })
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}, (names, cosine_similarity(emb_matrix))

# Threshold: total verses > the largest surah (286) -- direct complement of
# the earlier "size-matched" (minor-poet) filter.
max_surah_verses = max(surah_verse_count.values())
large_poets = {p: v for p, v in poet_total_verses.items() if v > max_surah_verses}

print(f"Threshold: total verses > {max_surah_verses}")
print(f"Well-documented poets qualifying: {len(large_poets)} of {len(poet_total_verses)}")
print(f"Their verse-count range: {min(large_poets.values())} to {max(large_poets.values())}")

exp7_embeddings = {p: poet_embeddings[p] for p in large_poets if p in poet_embeddings}
exp7_embeddings["Quran (whole)"] = quran_whole_vector
exp7_types = {nm: ("Quran" if nm == "Quran (whole)" else "Poet") for nm in exp7_embeddings}

exp7_df, exp7_metrics, (exp7_names, exp7_sim) = cluster_and_report(
    exp7_embeddings, exp7_types,
    f"Experiment 7: {len(large_poets)} well-documented poets + Quran (whole)")

quran_row7 = exp7_df[exp7_df["name"] == "Quran (whole)"].iloc[0]
same_cluster7 = exp7_df[(exp7_df["cluster"] == quran_row7["cluster"]) & (exp7_df["name"] != "Quran (whole)")]
print(f"\nQuran's cluster: {quran_row7['cluster']}")
print(f"Poets sharing it: {len(same_cluster7)}")
if len(same_cluster7) > 0:
    print(same_cluster7["name"].tolist())

qidx7 = exp7_names.index("Quran (whole)")
sims7 = exp7_sim[qidx7].copy(); sims7[qidx7] = -1
top5_idx7 = np.argsort(sims7)[-5:][::-1]
print("\nTop 5 most similar poets to the Quran:")
for i in top5_idx7:
    print(f"  {exp7_names[i]}: {sims7[i]:.4f}")

exp7_df.to_csv(TABLES_DIR / "experiment7_large_poets_only.csv", index=False)

Threshold: total verses > 286
Well-documented poets qualifying: 35 of 260
Their verse-count range: 288 to 3495


TypeError: Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.

In [ ]:
# CELL 8 -- Compare across all poet-pool variants tried so far
comparison_rows = [
    {"variant": "All 260 poets (original Exp 1)", "n_poets": 260,
     "quran_cluster": "Cluster 1 (19 poets)", "top_similarity": "0.976"},
    {"variant": "241 poets (19 closest excluded)", "n_poets": 241,
     "quran_cluster": "Outlier (48 poets)", "top_similarity": "0.972"},
    {"variant": f"{len(large_poets)} well-documented poets only (this run)", "n_poets": len(large_poets),
     "quran_cluster": f"Cluster {quran_row7['cluster']} ({len(same_cluster7)} poets)" if quran_row7['cluster'] != -1 else "Outlier",
     "top_similarity": f"{sims7[top5_idx7[0]]:.4f}"},
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLES_DIR / "poet_pool_comparison.csv", index=False)
print(comparison_df.to_string(index=False))

In [ ]:
# CELL 9 -- Figure
fig, ax = plt.subplots(figsize=(12, 10))
unique_clusters = sorted(exp7_df["cluster"].unique())
n_clust_plot = len([c for c in unique_clusters if c >= 0])
colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))

for cl in unique_clusters:
    sub = exp7_df[exp7_df["cluster"] == cl]
    for t, marker, size, edge, lw in [("Poet", "o", 60, "black", 0.4), ("Quran", "*", 400, "red", 1.8)]:
        tsub = sub[sub["type"] == t]
        if len(tsub) == 0:
            continue
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        ax.scatter(tsub["umap_x"], tsub["umap_y"], c=[color], marker=marker, s=size,
                  alpha=0.85, edgecolors=edge, linewidth=lw)

ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
ax.set_title(f"Experiment 7: {len(large_poets)} Well-Documented Poets + Quran (whole)\n(star=Quran, circles=poets)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "experiment7_umap.png", dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
# CELL 10 -- Final report
report_path = REPORTS_DIR / "experiment7_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXPERIMENT 7: WELL-DOCUMENTED POETS ONLY + QURAN\n")
    f.write("=" * 70 + "\n\n")

    f.write("METHOD\n" + "-" * 40 + "\n")
    f.write(f"  Threshold: total verses > {max_surah_verses} (largest surah's verse count)\n")
    f.write(f"  Qualifying poets: {len(large_poets)} of {len(poet_total_verses)}\n")
    f.write(f"  Their verse-count range: {min(large_poets.values())} to {max(large_poets.values())}\n\n")

    f.write("RESULT\n" + "-" * 40 + "\n")
    for k, v in exp7_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"  Quran's cluster: {quran_row7['cluster']}\n")
    f.write(f"  Poets sharing it: {len(same_cluster7)}\n")
    if len(same_cluster7) > 0:
        f.write(f"    {same_cluster7['name'].tolist()}\n")
    f.write("  Top 5 nearest poets:\n")
    for i in top5_idx7:
        f.write(f"    {exp7_names[i]}: {sims7[i]:.4f}\n")

    f.write("\nCOMPARISON ACROSS POET-POOL VARIANTS\n" + "-" * 40 + "\n")
    f.write(comparison_df.to_string(index=False) + "\n")

    f.write("\nHOW TO READ THIS\n" + "-" * 40 + "\n")
    f.write("  This is the complement of the earlier size-matched (minor-poet)\n")
    f.write("  test, which found MORE overlap when restricted to minor poets.\n")
    f.write("  If overlap here (well-documented poets only) is markedly LOWER\n")
    f.write("  than the original all-260 result, that supports the sample-size\n")
    f.write("  explanation: minor poets were driving the earlier overlap. If\n")
    f.write("  overlap persists here too, that argues against it -- meaning\n")
    f.write("  something other than sample size explains the pattern.\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment7_large_poets_only.csv` — full cluster assignment
- `output/tables/poet_pool_comparison.csv` — side-by-side comparison across
  all three poet-pool variants tried so far
- `output/figures/experiment7_umap.png` — UMAP scatter
- `output/reports/experiment7_report.txt` — full summary

Send the report + comparison table for the clearest before/after picture.